In [1]:
import os
import oracledb
from dotenv import load_dotenv
import pandas as pd

load_dotenv()

def create_cursor():
    connection = oracledb.connect(
        user=os.getenv("ORACLE_USER"),
        password=os.getenv("ORACLE_PASSWORD"),
        dsn=oracledb.makedsn("localhost", 1522, service_name="stu"),
    )
    cursor = connection.cursor()
    print("Connected to Oracle")
    return connection, cursor

# RQ3: Does YouTube comment sentiment reflect voter preferences or amplify extreme views?

In [2]:
connection, cursor = create_cursor()

rq3_query = """
SELECT
ch.channel_name,
v.lean,
v.year,
ROUND(AVG(c.sentiment_score), 3) AS avg_sentiment,
COUNT(*) AS comment_count
FROM youtube_comments c
JOIN youtube_videos v ON c.video_id = v.video_id
JOIN youtube_channels ch ON v.channel_id = ch.channel_id
GROUP BY ch.channel_name, v.lean, v.year
ORDER BY v.year, v.lean, ch.channel_name
"""

cursor.execute(rq3_query)
results = cursor.fetchall()
rq3_df = pd.DataFrame(results, columns=[d[0] for d in cursor.description])
display(rq3_df.head())

cursor.close()
connection.close()

Connected to Oracle


,CHANNEL_NAME,LEAN,YEAR,AVG_SENTIMENT,COMMENT_COUNT
0,Ben Shapiro,conservative,2018,0.229,108
1,The Young Turks,liberal,2018,-0.025,5
2,Ben Shapiro,conservative,2019,0.240,108
3,The Young Turks,liberal,2019,-0.028,2
4,Ben Shapiro,conservative,2020,0.043,108


In [3]:
fig, ax = plt.subplots(figsize=(12, 5))

channel_colors = {
    'Ben Shapiro': 'red',
    'The Young Turks': 'blue',
}
channel_styles = {
    'Ben Shapiro':    '-',
    'The Young Turks': '--',
}

for channel, group in rq3_df.groupby('CHANNEL_NAME'):
    group = group.sort_values('YEAR')
    ax.plot(
        group['YEAR'],
        group['AVG_SENTIMENT'],
        label=channel,
        color=channel_colors[channel],
        linestyle=channel_styles[channel],
    )

ax.set_title('Avg comment sentiment over time by channel')
ax.set_xlabel('Year')
ax.set_ylabel('Avg Sentiment Score ')
ax.legend()
ax.grid(alpha=0.3)
plt.savefig('rq3_sentiment.png')
plt.show()

NameError: name 'plt' is not defined